# Building a Data Dictionary for COUNTRY_BANK_DEMO_DB

A **data dictionary** documents every schema, table, and column in your database — including data types, descriptions, and business context. This notebook walks through:

## Summary

| Step | What It Does |
|------|--------------|
| 1 | Discover schemas in the database |
| 2 | List all tables/views with row counts |
| 3 | Extract column-level metadata (types, nullability) |
| 4 | Add business descriptions via `COMMENT ON` |
| 5 | Create a reusable `DATA_DICTIONARY` view |
| 6 | Query the dictionary for any table |
| 7 | Track documentation coverage gaps |

## Step 1: Discover Schemas

First, let's see what schemas exist in `COUNTRY_BANK_DEMO_DB` (excluding the system `INFORMATION_SCHEMA`).

In [ ]:
%%sql -r schemas
SELECT
    CATALOG_NAME AS DATABASE_NAME,
    SCHEMA_NAME,
    SCHEMA_OWNER,
    CREATED,
    LAST_ALTERED,
    COMMENT
FROM COUNTRY_BANK_DEMO_DB.INFORMATION_SCHEMA.SCHEMATA
WHERE SCHEMA_NAME != 'INFORMATION_SCHEMA'
ORDER BY SCHEMA_NAME;

## Step 2: List All Tables and Views

Query `INFORMATION_SCHEMA.TABLES` to get every table and view, along with row counts and any existing comments.

In [ ]:
%%sql -r tables_list
SELECT
    TABLE_SCHEMA,
    TABLE_NAME,
    TABLE_TYPE,
    ROW_COUNT,
    BYTES,
    CREATED,
    LAST_ALTERED,
    COMMENT AS TABLE_COMMENT
FROM COUNTRY_BANK_DEMO_DB.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA != 'INFORMATION_SCHEMA'
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## Step 3: Extract Column-Level Details

This is the core of the data dictionary — every column with its data type, position, nullability, and any existing description.

In [ ]:
%%sql -r columns_detail
SELECT
    TABLE_SCHEMA,
    TABLE_NAME,
    ORDINAL_POSITION,
    COLUMN_NAME,
    DATA_TYPE,
    IS_NULLABLE,
    COLUMN_DEFAULT,
    CHARACTER_MAXIMUM_LENGTH,
    NUMERIC_PRECISION,
    NUMERIC_SCALE,
    COMMENT AS COLUMN_COMMENT
FROM COUNTRY_BANK_DEMO_DB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA != 'INFORMATION_SCHEMA'
ORDER BY TABLE_SCHEMA, TABLE_NAME, ORDINAL_POSITION;

## Step 4: Add Business Descriptions with COMMENT ON

Use `COMMENT ON` to enrich your tables and columns with business context. Here's an example for the `HOME_POLICY_FREQ` table:

In [ ]:
%%sql -r comment_table_result
COMMENT ON TABLE COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.HOME_POLICY_FREQ IS
    'Home insurance policy frequency table containing claim counts, exposure, and risk factors for actuarial pricing models.';

### Adding Column-Level Descriptions

Now we use `COMMENT ON COLUMN` to attach human-readable business definitions to individual columns. These comments are stored in Snowflake metadata and will appear in `INFORMATION_SCHEMA.COLUMNS`, `DESCRIBE TABLE`, and our `DATA_DICTIONARY` view. This is how you make columns self-documenting for anyone querying the database.

In [ ]:
%%sql -r comment_columns_result
COMMENT ON COLUMN COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.HOME_POLICY_FREQ.CLAIM_COUNT IS
    'Number of claims filed on this policy during the exposure period.';

COMMENT ON COLUMN COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.HOME_POLICY_FREQ.EXPOSURE IS
    'Duration of policy coverage in years (e.g., 0.5 = 6 months). Used as offset in frequency models.';

COMMENT ON COLUMN COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.HOME_POLICY_FREQ.LOSS_HISTORY_SCORE IS
    'Numeric score representing the policyholder prior loss history. Higher values indicate more prior claims.';

## Step 5: Build a Unified Data Dictionary View

Create a reusable view that joins table and column metadata into a single queryable data dictionary.

In [ ]:
%%sql -r create_view_result
CREATE OR REPLACE VIEW COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.DATA_DICTIONARY AS
SELECT
    t.TABLE_SCHEMA,
    t.TABLE_NAME,
    t.TABLE_TYPE,
    t.ROW_COUNT,
    t.COMMENT AS TABLE_DESCRIPTION,
    c.ORDINAL_POSITION AS COL_POSITION,
    c.COLUMN_NAME,
    c.DATA_TYPE,
    c.IS_NULLABLE,
    c.CHARACTER_MAXIMUM_LENGTH AS MAX_LENGTH,
    c.NUMERIC_PRECISION,
    c.NUMERIC_SCALE,
    c.COLUMN_DEFAULT,
    c.COMMENT AS COLUMN_DESCRIPTION
FROM COUNTRY_BANK_DEMO_DB.INFORMATION_SCHEMA.TABLES t
JOIN COUNTRY_BANK_DEMO_DB.INFORMATION_SCHEMA.COLUMNS c
    ON t.TABLE_SCHEMA = c.TABLE_SCHEMA
    AND t.TABLE_NAME = c.TABLE_NAME
WHERE t.TABLE_SCHEMA != 'INFORMATION_SCHEMA'
ORDER BY t.TABLE_SCHEMA, t.TABLE_NAME, c.ORDINAL_POSITION;

## Step 6: Query the Data Dictionary

Now you can query the dictionary view anytime for documentation, audits, or onboarding.

In [ ]:
%%sql -r dictionary_example
SELECT *
FROM COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.DATA_DICTIONARY
WHERE TABLE_NAME = 'HOME_POLICY_FREQ'
ORDER BY COL_POSITION;

In [ ]:
print("DATA DICTIONARY: HOME_POLICY_FREQ")
print("=" * 80)
for _, row in dictionary_example.iterrows():
    desc = row.get('COLUMN_DESCRIPTION', '') or '(no description yet)'
    nullable = 'NULL' if row['IS_NULLABLE'] == 'YES' else 'NOT NULL'
    print(f"  {row['COL_POSITION']:>2}. {row['COLUMN_NAME']:<30} {row['DATA_TYPE']:<15} {nullable:<10} {desc}")

## Step 7: Find Undocumented Columns

Identify columns that still need business descriptions — useful for tracking documentation coverage.

In [ ]:
%%sql -r undocumented_cols
SELECT
    TABLE_SCHEMA,
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE
FROM COUNTRY_BANK_DEMO_DB.ACTUARIAL_PRICING.DATA_DICTIONARY
WHERE COLUMN_DESCRIPTION IS NULL
ORDER BY TABLE_SCHEMA, TABLE_NAME, COL_POSITION;